# Matmul算子开发

## 概述

矩阵乘法（Matmul）是深度学习和科学计算中最核心的算子之一，在LLM大模型中占计算量的90%以上。本章将系统讲解如何使用pyasc在昇腾NPU上开发Matmul算子，从基础Cube指令到高阶API，从单核到多核并行，覆盖Cube Only和MIX两种执行模式。

本章以"基础API体会硬件机制 → 高阶API解放生产力 → 两种模式实战 → 综合实践"的路线由浅入深组织教学。

<img src="./images/04.01_chapter_intro/chapter_roadmap.png" alt="本章学习路线" width="700px">

## 什么是Matmul算子

Matmul（矩阵乘法）计算公式为 **C = A × B**（可选加偏置 **C = A × B + Bias**），其中：

- **A** 为左矩阵，形状 `[M, K]`
- **B** 为右矩阵，形状 `[K, N]`
- **C** 为结果矩阵，形状 `[M, N]`
- **Bias** 为偏置向量，形状 `[1, N]`

每个元素的值由三层循环计算：`C[i,j] = Σ(A[i,k] × B[k,j]) + Bias[j]`

如下图所示，结果矩阵的每个元素 `C[i,j]` 是A的第i行与B的第j列沿K轴做点积（逐元素相乘后求和）的结果。K维度是A和B的公共维度，也称为"收缩维度"。

<img src="./images/04.01_chapter_intro/matmul_concept.png" alt="矩阵乘法概念" width="600px">

在昇腾NPU中，Matmul由专用的**Cube计算单元**执行，与Vector算子（Add、ReLU等）使用不同的硬件单元。Cube计算单元通过mmad指令一个cycle可以完成一个分形（fp16下16×16）的矩阵乘加，相比逐元素计算效率高出数个量级。理解Cube计算的存储层级和数据搬运机制，是开发高性能Matmul算子的基础。

## 学习前置要求

在开始本章学习前，请确认你已具备以下能力：

| 类别 | 要求 |
| --- | --- |
| **已具备知识** | 理解矩阵乘法的数学原理；了解pyasc的基本编程范式（Host-Device异构、SPMD并行） |
| **前置章节** | 已完成第1章（pyasc概述）和第2章（核函数基础）的学习 |
| **环境要求** | pyasc v1.1.0 及以上；CANN 8.5.0.alpha001 及以上；Python 3.9-3.12 |
| **硬件要求** | Atlas A2（910B）或 Atlas A3（910C）AI处理器 |

> 本章涉及Cube存储层级、分形格式等底层概念，如果你尚未接触过昇腾NPU的硬件架构，建议先回顾第1章的架构介绍。

## 学习目标

完成本章后，你将能够：

1. **理解Cube存储层级**：掌握GM→L1→L0A/L0B→L0C→GM的数据流路径和各级存储的职责
2. **掌握基础mmad指令**：使用`asc.data_copy`/`asc.load_data`/`asc.mmad`/`asc.fixpipe`从零实现矩阵乘法
3. **掌握高阶Matmul API**：使用`asc.adv.Matmul`体系快速开发多核并行Matmul算子
4. **理解Cube Only模式**：通过`matmul_cube_only=True`实现纯Cube多核计算
5. **理解MIX模式**：掌握AIC+AIV协同工作机制和多核偏移计算
6. **独立开发Matmul算子**：综合运用上述知识完成带Bias的MIX模式Matmul实践

## 章节内容导航

| 小节 | 主题 | 核心内容 |
| --- | --- | --- |
| [4.1 章节概述](./04.01_chapter_intro.ipynb) | 本章导览 | 学习前置要求、目标、内容导航 |
| [4.2 基础Matmul实现](./04.02_matmul_basics_and_basic_api.ipynb) | Matmul基础与基础API | Cube存储层级、分形格式、K方向分块累加、`asc.data_copy`/`asc.load_data`/`asc.mmad`/`asc.fixpipe` |
| [4.3 高阶Matmul API](./04.03_advanced_matmul_api.ipynb) | 高阶API体系与实操 | `asc.adv.Matmul`/`MatmulType`/`register_matmul`、Tiling详解、基础vs高阶对比 |
| [4.4 Cube Only与MIX模式](./04.04_cube_only_and_mix_mode_matmul.ipynb) | 两种执行模式开发 | Cube Only（`matmul_cube_only=True`、Bias）与MIX（AIC+AIV、核数`//2`）模式实现、`calc_offsets`多核偏移、选型策略 |
| [4.5 章节实践](./04.05_chapter_practice.ipynb) | 综合编程实践 | 带Bias的MIX模式Matmul算子开发（TODO模板+参考答案） |

## 运行环境与硬件说明

### 支持硬件

| 硬件型号 | 架构 | 支持状态 |
| --- | --- | --- |
| Atlas A2（910B） | dav-matrix-core | ✅ 完全支持 |
| Atlas A3（910C） | dav-matrix-core | ✅ 完全支持 |

### 体验环境

- **在线体验**：learning-hub notebook 在线体验环境
- **云开发**：CANNLab 910B/910C 云开发环境

### 运行方式

本章所有代码通过 `%%writefile` 写入 `.py` 文件后，使用 `!python3 Sources/04.0X/xxx.py -r NPU` 运行验证。每个notebook从上到下依次执行即可完成全部学习。

> **注意**：代码中的 `USE_CORE_NUM` 参数需根据实际硬件核数调整。910B 通常为 24 核（AIC）/48 核（AIV），910C 为 32 核（AIC）/64 核（AIV）。

---

下一节：[4.2 基础mmad实现](./04.02_matmul_basics_and_low_level_api.ipynb)